In [1]:
import pandas as pd
import numpy as np

data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [2]:
data.drop(['RowNumber','CustomerId','Surname'],axis=1,inplace=True)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
data.Tenure.value_counts()

Tenure
2     1048
1     1035
7     1028
8     1025
5     1012
3     1009
4      989
9      984
6      967
10     490
0      413
Name: count, dtype: int64

In [4]:
min(data.EstimatedSalary.values),max(data.EstimatedSalary.values)

(11.58, 199992.48)

In [5]:
data.IsActiveMember.value_counts()

IsActiveMember
1    5151
0    4849
Name: count, dtype: int64

In [6]:
data.HasCrCard.value_counts()

HasCrCard
1    7055
0    2945
Name: count, dtype: int64

In [7]:
min(data.Balance.values),max(data.Balance.values)

(0.0, 250898.09)

In [8]:
data.NumOfProducts.value_counts()

NumOfProducts
1    5084
2    4590
3     266
4      60
Name: count, dtype: int64

In [9]:
min(data.Age.values),max(data.Age.values)

(18, 92)

In [10]:
data.Geography.value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [11]:
min(data.CreditScore.values),max(data.CreditScore.values)

(350, 850)

In [12]:
data.isnull().sum()

CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [13]:
data.duplicated().sum()

0

In [14]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  object 
 2   Gender           10000 non-null  object 
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


In [15]:
from sklearn.model_selection import train_test_split

x=data.drop('Exited',axis=1)
y=data['Exited']

x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
x_train,x_test,y_train,y_test

(      CreditScore Geography  Gender  Age  Tenure    Balance  NumOfProducts  \
 9254          686    France    Male   32       6       0.00              2   
 1561          632   Germany    Male   42       4  119624.60              2   
 1670          559     Spain    Male   24       3  114739.92              1   
 6087          561    France  Female   27       9  135637.00              1   
 6669          517    France    Male   56       9  142147.32              1   
 ...           ...       ...     ...  ...     ...        ...            ...   
 5734          768    France    Male   54       8   69712.74              1   
 5191          682    France  Female   58       1       0.00              1   
 5390          735    France  Female   38       1       0.00              3   
 860           667    France    Male   43       8  190227.46              1   
 7270          697   Germany    Male   51       1  147910.30              1   
 
       HasCrCard  IsActiveMember  EstimatedSalary 

In [16]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ct=ColumnTransformer(transformers=[('onehot', OneHotEncoder(drop='first'), ['Geography', 'Gender'])], remainder='passthrough')

x_train=ct.fit_transform(x_train)
x_test=ct.transform(x_test)

In [17]:
from sklearn.preprocessing import StandardScaler

s=StandardScaler()
x_train=s.fit_transform(x_train)
x_test=s.transform(x_test)

In [18]:
import pickle

with open('columntransformer.pkl', 'wb') as file:
    pickle.dump(ct,file)

with open('standardscaler.pkl', 'wb') as file:
    pickle.dump(s,file)

In [19]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [20]:
model=Sequential([
    Dense(64,activation='relu',input_dim=x_train.shape[1]),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
])

model.summary()

c:\Users\sekha\Desktop\churn prediction\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [22]:
log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [23]:
earlystopping_callback=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [24]:
y_train.shape

(8000,)

In [25]:
history=model.fit(
    x_train,y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[earlystopping_callback,tensorboard_callback]
)

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8050 - loss: 0.4520 - val_accuracy: 0.8306 - val_loss: 0.4109
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8327 - loss: 0.4051 - val_accuracy: 0.8369 - val_loss: 0.3880
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8502 - loss: 0.3727 - val_accuracy: 0.8462 - val_loss: 0.3677
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8587 - loss: 0.3523 - val_accuracy: 0.8494 - val_loss: 0.3547
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8598 - loss: 0.3426 - val_accuracy: 0.8512 - val_loss: 0.3492
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8597 - loss: 0.3373 - val_accuracy: 0.8537 - val_loss: 0.3456
Epoch 7/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8634 - loss: 0.3321 - val_accuracy: 0.8525 - val_loss: 0.3527
Epoch 8/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8650 - loss: 0.3306 - val_accu

In [26]:
model.save('ann.keras')

In [27]:
!kill 2708

'kill' is not recognized as an internal or external command,
operable program or batch file.


In [28]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

Launching TensorBoard...

In [29]:
from tensorflow.keras.models import load_model

predict_model=load_model('ann.keras')

with open('columntransformer.pkl', 'rb') as file:
    coltrans=pickle.load(file)

with open('standardscaler.pkl', 'rb') as file:
    stdscaler=pickle.load(file)

In [30]:
input_data= {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [31]:
df_input=pd.DataFrame([input_data])
df_input=coltrans.transform(df_input)
df_input=stdscaler.transform(df_input)
df_input

array([[-0.57946723, -0.57638802,  0.91324755, -0.53598516,  0.10479359,
        -0.69539349, -0.25781119,  0.80843615,  0.64920267,  0.97481699,
        -0.87683221]])

In [32]:
predict_model.predict(df_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step


array([[0.04648365]], dtype=float32)